**Objectives**
1. Clean the data.
2. Map Encoding for the required columns to numeric coulmns
3. Binning data columns if require
4. Save the cleaned data csv.

#Setup

Using the following libraries

[pandas] for managing the data.

[numpy] for mathematical operations.

[matplotlib] for additional plotting tools.

Import all required libraries in one place_


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
#mount the drive to upload the data
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


###Encoding for ML Model Development


In [ ]:
df1 = pd.read_csv('/content/drive/MyDrive/LaptopsPrice prediction data/laptops_test.csv')
df2 = pd.read_csv('/content/drive/MyDrive/LaptopsPrice prediction data/laptops_train.csv')
df = pd.concat([df1, df2], ignore_index=True).copy()

In [ ]:
df.head()

,Manufacturer,Model Name,Category,Screen Size,Screen,CPU,RAM,Storage,GPU,Operating System,Operating System Version,Weight,Price
0,HP,15-bs053od (i7-7500U/6GB/1TB/W10),Notebook,"15.6""",1366x768,Intel Core i7 7500U 2.7GHz,6GB,1TB HDD,Intel HD Graphics 620,Windows,10,2.04kg,5148468.0
1,Asus,Rog GL753VE-DS74,Gaming,"17.3""",Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,Nvidia GeForce GTX 1050 Ti,Windows,10,2.99kg,15552108.0
2,Dell,Inspiron 7579,2 in 1 Convertible,"15.6""",IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 7500U 2.7GHz,12GB,512GB SSD,Intel HD Graphics 620,Windows,10,2.19kg,11550708.0
3,Toshiba,Portege Z30-C-1CV,Notebook,"13.3""",Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows,7,1.2kg,10625940.0
4,Lenovo,IdeaPad 320-15ABR,Notebook,"15.6""",Full HD 1920x1080,AMD A12-Series 9720P 3.6GHz,6GB,256GB SSD,AMD Radeon 530,Windows,10,2.2kg,4881708.0


In [ ]:
df["GPU"]

,GPU
0,Intel HD Graphics 620
1,Nvidia GeForce GTX 1050 Ti
2,Intel HD Graphics 620
3,Intel HD Graphics 520
4,AMD Radeon 530
...,...
1297,Nvidia GeForce GTX 1070
1298,Intel HD Graphics 520
1299,Nvidia GeForce GTX 1060
1300,Nvidia GeForce 930MX


In [ ]:
def map_gpu(gpu):
    gpu = str(gpu)

    if "AMD" in gpu:
        return 1
    elif "Intel" in gpu:
        return 2
    elif "Nvidia" in gpu or "NVIDIA" in gpu:
        return 3
    else:
        return None  # or 0

df["GPU"] = df["GPU"].apply(map_gpu)

In [ ]:
df["GPU"].isnull().sum()

np.int64(1)

In [ ]:
df["GPU"] = df["GPU"].fillna(0).astype(int)

In [ ]:
df["GPU"] = df["GPU"].astype(int)

In [ ]:
#From CPU column we need to seprate frquency of the CPU in GHz.
df["CPU_frequency"] = df["CPU"].str.extract(r'(\d+\.\d+|\d+)GHz').astype(float)

In [ ]:
#As the CPU_frequency has categorical variable so we need to change it into numerical for further use in ML and to fill the missing data.
df["CPU_frequency"] = pd.to_numeric(df["CPU_frequency"], errors='coerce')

In [ ]:
#From CPU column we need to sepearet the CPU_Core.
df["CPU_core"] = df["CPU"].str.extract(r'(i[3579])')

In [ ]:
#for Storage column we need only SSD data so we will extarct the data from Storage Column and create a new Column Storage_GB_SSD.
df["Storage_GB_SSD"] = df[' Storage'].str.extract(r'(\d+)\s*GB\s*SSD')

In [ ]:
#First we will map the CPU_core into Numeric form.
df["CPU_core"] = df["CPU_core"].map({
    "i3": 3,
    "i5": 5,
    "i7": 7
})

In [ ]:
#use median to fill the null values as this is important column we can not just drop this.
df["CPU_core"] = df["CPU_core"].fillna(df["CPU_core"].median())

In [ ]:
#map the Storage_GB_SSD into Numeric form.

df["Storage_GB_SSD"] = pd.to_numeric(df["Storage_GB_SSD"], errors="coerce")

In [ ]:
#use median to fill the null values.
df["Storage_GB_SSD"] = df["Storage_GB_SSD"].fillna(df["Storage_GB_SSD"].median())

In [ ]:
#df = df.drop("Operating System Version", axis=1)

In [ ]:
df["CPU_core"] = df["CPU_core"].astype(int)

In [ ]:
df["Storage_GB_SSD"] = df["Storage_GB_SSD"].astype(int)

In [ ]:
df["Price"] = df["Price"].astype(int)

In [ ]:
#Lets Encode the columns
# Category Mapping
category_map = {
    "Gaming": 1,
    "Netbook": 2,
    "Notebook": 3,
    "Ultrabook": 4,
    "Workstation": 5
}

df["Category"] = df["Category"].map(category_map)

In [ ]:
df.head()

,Manufacturer,Model Name,Category,Screen Size,Screen,CPU,RAM,Storage,GPU,Operating System,Operating System Version,Weight,Price,CPU_frequency,CPU_core,Storage_GB_SSD
0,HP,15-bs053od (i7-7500U/6GB/1TB/W10),3.0,"15.6""",1366x768,Intel Core i7 7500U 2.7GHz,6GB,1TB HDD,2,Windows,10,2.04kg,5148468,2.7,7,256
1,Asus,Rog GL753VE-DS74,1.0,"17.3""",Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,3,Windows,10,2.99kg,15552108,2.8,7,256
2,Dell,Inspiron 7579,NaN,"15.6""",IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 7500U 2.7GHz,12GB,512GB SSD,2,Windows,10,2.19kg,11550708,2.7,7,512
3,Toshiba,Portege Z30-C-1CV,3.0,"13.3""",Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,2,Windows,7,1.2kg,10625940,2.3,5,128
4,Lenovo,IdeaPad 320-15ABR,3.0,"15.6""",Full HD 1920x1080,AMD A12-Series 9720P 3.6GHz,6GB,256GB SSD,1,Windows,10,2.2kg,4881708,3.6,5,256


In [ ]:
df.isnull().sum()

,0
Manufacturer,0
Model Name,0
Category,120
Screen Size,0
Screen,0
CPU,0
RAM,0
Storage,0
GPU,0
Operating System,0


In [ ]:
df["Category"] = df["Category"].fillna(df["Category"].mode()[0])

In [ ]:
df.drop("Operating System Version", axis=1, inplace=True)

In [ ]:
df.drop("Model Name", axis=1, inplace=True)

In [ ]:
df.isnull().sum()

,0
Manufacturer,0
Category,0
Screen Size,0
Screen,0
CPU,0
RAM,0
Storage,0
GPU,0
Operating System,0
Weight,0


In [ ]:
df.dtypes

,0
Manufacturer,object
Category,float64
Screen Size,object
Screen,object
CPU,object
RAM,object
Storage,object
GPU,int64
Operating System,object
Weight,object


In [ ]:
df.columns

Index(['Manufacturer', 'Category', 'Screen Size', 'Screen', 'CPU', 'RAM',
       ' Storage', 'GPU', 'Operating System', 'Weight', 'Price',
       'CPU_frequency', 'CPU_core', 'Storage_GB_SSD'],
      dtype='object')

In [ ]:
df["IPS"] = df["Screen"].str.contains("IPS").astype(int)
df["Touchscreen"] = df["Screen"].str.contains("Touch").astype(int)
df["Full_HD"] = df["Screen"].str.contains("Full HD").astype(int)
df["HD_1920"] = df["Screen"].str.contains("1920x1080").astype(int)

In [ ]:
df["Price-binned"] = pd.cut(
    df["Price"],
    bins=3,
    labels=["Low", "Medium", "High"]
)

In [ ]:
df

,Manufacturer,Category,Screen Size,Screen,CPU,RAM,Storage,GPU,Operating System,Weight,Price,CPU_frequency,CPU_core,Storage_GB_SSD,IPS,Touchscreen,Full_HD,HD_1920,Price-binned
0,HP,3.0,"15.6""",1366x768,Intel Core i7 7500U 2.7GHz,6GB,1TB HDD,2,Windows,2.04kg,5148468,2.7,7,256,0,0,0,0,Low
1,Asus,1.0,"17.3""",Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,3,Windows,2.99kg,15552108,2.8,7,256,0,0,1,1,Low
2,Dell,3.0,"15.6""",IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 7500U 2.7GHz,12GB,512GB SSD,2,Windows,2.19kg,11550708,2.7,7,512,1,1,1,1,Low
3,Toshiba,3.0,"13.3""",Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,4GB,128GB SSD,2,Windows,1.2kg,10625940,2.3,5,128,0,0,1,1,Low
4,Lenovo,3.0,"15.6""",Full HD 1920x1080,AMD A12-Series 9720P 3.6GHz,6GB,256GB SSD,1,Windows,2.2kg,4881708,3.6,5,256,0,0,1,1,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1297,Dell,1.0,"17.3""",Full HD 1920x1080,Intel Core i7 6700HQ 2.6GHz,32GB,256GB SSD + 1TB HDD,3,Windows,4.42kg,24897600,2.6,7,256,0,0,1,1,Medium
1298,Toshiba,3.0,"14.0""",Full HD 1920x1080,Intel Core i5 6200U 2.3GHz,8GB,256GB SSD,2,Windows,1.95kg,10492560,2.3,5,256,0,0,1,1,Low
1299,Asus,1.0,"17.3""",Full HD 1920x1080,Intel Core i7 7700HQ 2.8GHz,16GB,256GB SSD + 1TB HDD,3,Windows,2.73kg,18227710,2.8,7,256,0,0,1,1,Low
1300,HP,3.0,"15.6""",IPS Panel Full HD 1920x1080,Intel Core i5 7200U 2.70GHz,8GB,128GB SSD + 1TB HDD,3,Windows,2.04kg,8705268,2.7,5,128,1,0,1,1,Low


In [ ]:
#ename columns properly
df = df.rename(columns={
    "Screen Size": "Screen_Size_inch",
    "Operating System": "OS",
    "RAM": "RAM_GB",
    "Weight": "Weight_pounds"
})

In [ ]:
df.columns

Index(['Manufacturer', 'Category', 'Screen_Size_inch', 'Screen', 'CPU',
       'RAM_GB', ' Storage', 'GPU', 'OS', 'Weight_pounds', 'Price',
       'CPU_frequency', 'CPU_core', 'Storage_GB_SSD', 'IPS', 'Touchscreen',
       'Full_HD', 'HD_1920', 'Price-binned'],
      dtype='object')

In [ ]:
df["Screen_Size_inch"]= df["Screen_Size_inch"].astype(str).str.extract(r"(\d+\.?\d*)").astype(float)

In [ ]:
#Keep ONLY required columns
df = df[[
    "Manufacturer",
    "Category",
    "GPU",
    "OS",
    "CPU_core",
    "Screen_Size_inch",
    "CPU_frequency",
    "RAM_GB",
    "Storage_GB_SSD",
    "Weight_pounds",
    "Price",
    "IPS",
    "Touchscreen",
    "Full_HD",
    "HD_1920",
    "Price-binned"
]]

In [ ]:
df.to_csv("clean_laptop_data.csv", index=False)

In [ ]:
import os
os.listdir()

['.config', 'clean_laptop_data.csv', 'drive', 'sample_data']

In [ ]:
df.to_csv("/content/drive/MyDrive/laptop_clean.csv", index=False)